# M1 Normal vs Event Gate Policy Iteration

## tl;dr

17번 `normal vs event` gate가 `binary_gate_unstable`로 나왔기 때문에, 이번 노트북은 모델 고도화가 아니라 gate 기준을 반복 검증한다.

비교 대상은 `reference_pre_event_all`, `reference_low_overlap_hybrid`, `strict_no_overlap`, `low_overlap_only`, `train_normal_expanded`이다. 판단 기준은 event recall, normal FPR, Dummy 대비 balanced accuracy lift, group CV 누수 여부다.

## Context & Methods

- 최종 목표는 `normal / fault / task / activity` 4분류지만, 지금은 1단계 `normal vs event` gate 안정화가 목적이다.
- `event = fault + task + activity`이다.
- 기존 `normal vs pre_event` 방식처럼 기준을 세우고 검증한 뒤, 기준 수정 여부를 판단한다.
- 학습 feature는 `compact13`과 `expanded154`를 비교한다.
- `candidate_normal` 증강은 13번 산출물이 compact13만 보유하므로 compact13에서만 수행한다.

In [1]:

from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "07_데이터산출물"

LABEL_INDEX_PATH = OUTPUT_DIR / "m1_4class_label_candidate_index.csv"
WINDOW_AUDIT_PATH = OUTPUT_DIR / "m1_4class_window_policy_audit.csv"
FEATURE_SET_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
CANDIDATE_NORMAL_PATH = OUTPUT_DIR / "m1_expansion_feature_pool.csv"
REFERENCE_DECISION_PATH = OUTPUT_DIR / "m1_4class_binary_gate_decision_matrix.csv"
REFERENCE_PREDICTION_PATH = OUTPUT_DIR / "m1_4class_binary_gate_predictions.csv"

OUT_POLICY_AUDIT = OUTPUT_DIR / "m1_normal_vs_event_gate_policy_audit.csv"
OUT_DATASET_SUMMARY = OUTPUT_DIR / "m1_normal_vs_event_gate_dataset_summary.csv"
OUT_CV_METRICS = OUTPUT_DIR / "m1_normal_vs_event_gate_cv_metrics.csv"
OUT_PREDICTIONS = OUTPUT_DIR / "m1_normal_vs_event_gate_predictions.csv"
OUT_THRESHOLD_METRICS = OUTPUT_DIR / "m1_normal_vs_event_gate_threshold_metrics.csv"
OUT_ERROR_AUDIT = OUTPUT_DIR / "m1_normal_vs_event_gate_error_audit.csv"
OUT_DECISION_MATRIX = OUTPUT_DIR / "m1_normal_vs_event_gate_decision_matrix.csv"
OUT_REPORT = OUTPUT_DIR / "18_M1_normal_vs_event_gate_policy_iteration_보고서.md"

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLDS = [0.4, 0.5, 0.6]
DEFAULT_THRESHOLD = 0.5

for path in [LABEL_INDEX_PATH, WINDOW_AUDIT_PATH, FEATURE_SET_PATH, REFERENCE_DECISION_PATH, REFERENCE_PREDICTION_PATH]:
    assert path.exists(), path

print("project_root:", PROJECT_ROOT)
print("output_dir:", OUTPUT_DIR)


project_root: C:\3rd_Project\HeatGridAgent
output_dir: C:\3rd_Project\HeatGridAgent\07_데이터산출물


## Data

16번 taxonomy와 17번 binary gate 결과를 기준선으로 사용한다.

In [2]:

label_index = pd.read_csv(LABEL_INDEX_PATH)
window_audit = pd.read_csv(WINDOW_AUDIT_PATH)
feature_sets = pd.read_csv(FEATURE_SET_PATH)
reference_decision = pd.read_csv(REFERENCE_DECISION_PATH)
reference_predictions = pd.read_csv(REFERENCE_PREDICTION_PATH)
candidate_pool = pd.read_csv(CANDIDATE_NORMAL_PATH) if CANDIDATE_NORMAL_PATH.exists() else pd.DataFrame()

print("label_index", label_index.shape)
print("window_audit", window_audit.shape)
print("reference_decision", reference_decision.shape)
print("candidate_pool", candidate_pool.shape)
print(feature_sets[["feature_set", "feature_count"]].to_string(index=False))


label_index (200, 190)
window_audit (298, 190)
reference_decision (60, 33)
candidate_pool (131, 32)
      feature_set  feature_count
           base70             70
      expanded154            154
compact13_overlap             13
   compact20_main             20
  compact27_union             27


In [3]:

def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def parse_feature_list(feature_set_name: str) -> list[str]:
    row = feature_sets.loc[feature_sets["feature_set"] == feature_set_name]
    assert len(row) == 1, f"missing feature set: {feature_set_name}"
    features = [feature for feature in str(row.iloc[0]["features"]).split("|") if feature]
    missing = sorted(set(features) - set(label_index.columns))
    assert not missing, f"missing features in label index: {missing[:10]}"
    return features


FEATURE_COLUMNS = {
    "compact13": parse_feature_list("compact13_overlap"),
    "expanded154": parse_feature_list("expanded154"),
}
assert len(FEATURE_COLUMNS["compact13"]) == 13
assert len(FEATURE_COLUMNS["expanded154"]) == 154
print({k: len(v) for k, v in FEATURE_COLUMNS.items()})


{'compact13': 13, 'expanded154': 154}


## Build Policy Candidate Datasets

reference, no-overlap, low-overlap, train-normal-expanded 후보를 구성한다.

In [4]:

def overlap_count(frame: pd.DataFrame) -> pd.Series:
    return pd.to_numeric(frame["overlap_disturbance_count"], errors="coerce").fillna(0)


def finalize_dataset(frame: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    data = frame.copy()
    data["dataset"] = dataset_name
    data["binary_label"] = np.where(data["final_class"].eq("normal"), 0, 1)
    data["binary_label_name"] = np.where(data["binary_label"].eq(0), "normal", "event")
    data["event_type_binary"] = np.where(data["final_class"].eq("normal"), "normal", data["final_class"])
    data["substation_id"] = data["substation_id"].astype(int)
    duplicate_ids = data["source_id"][data["source_id"].duplicated()].unique().tolist()
    assert not duplicate_ids, f"{dataset_name} duplicate source_id: {duplicate_ids[:10]}"
    assert data["binary_label"].nunique() == 2, dataset_name
    return data


def reference_pre_event_all() -> pd.DataFrame:
    return label_index.loc[as_bool(label_index["baseline_included"])].copy()


def reference_low_overlap_hybrid() -> pd.DataFrame:
    eligible = as_bool(window_audit["coverage_eligible"])
    parts = [
        window_audit.loc[(window_audit["final_class"] == "normal") & (window_audit["window_policy"] == "normal_event_7d") & eligible],
        window_audit.loc[(window_audit["final_class"] == "fault") & (window_audit["window_policy"] == "fault_pre_7d") & eligible],
        window_audit.loc[(window_audit["final_class"] == "task") & (window_audit["window_policy"] == "task_post_7d") & eligible],
        window_audit.loc[(window_audit["final_class"] == "activity") & (window_audit["window_policy"] == "activity_pre_7d") & eligible],
    ]
    return pd.concat(parts, ignore_index=True).copy()


def pick_priority_rows(frame: pd.DataFrame, class_name: str, policies: list[str], require_no_overlap: bool) -> pd.DataFrame:
    subset = frame.loc[frame["final_class"].eq(class_name) & as_bool(frame["coverage_eligible"])].copy()
    if require_no_overlap:
        subset = subset.loc[overlap_count(subset).eq(0)].copy()
    policy_rank = {policy: rank for rank, policy in enumerate(policies)}
    subset["policy_rank"] = subset["window_policy"].map(policy_rank)
    subset = subset.loc[subset["policy_rank"].notna()].copy()
    id_col = "disturbance_row_id" if class_name != "normal" else "source_event_id"
    subset = subset.sort_values([id_col, "policy_rank", "coverage_rate"], ascending=[True, True, False])
    return subset.drop_duplicates(subset=[id_col], keep="first").drop(columns=["policy_rank"])


def strict_no_overlap() -> pd.DataFrame:
    parts = [
        pick_priority_rows(window_audit, "normal", ["normal_event_7d"], require_no_overlap=False),
        pick_priority_rows(window_audit, "fault", ["fault_pre_7d"], require_no_overlap=True),
        pick_priority_rows(window_audit, "task", ["task_post_7d", "task_pre_7d"], require_no_overlap=True),
        pick_priority_rows(window_audit, "activity", ["activity_pre_7d", "activity_post_7d"], require_no_overlap=True),
    ]
    return pd.concat(parts, ignore_index=True).copy()


def low_overlap_only() -> pd.DataFrame:
    data = reference_low_overlap_hybrid()
    return data.loc[(data["final_class"].eq("normal")) | overlap_count(data).eq(0)].copy()


DATASETS = {
    "reference_pre_event_all": finalize_dataset(reference_pre_event_all(), "reference_pre_event_all"),
    "reference_low_overlap_hybrid": finalize_dataset(reference_low_overlap_hybrid(), "reference_low_overlap_hybrid"),
    "strict_no_overlap": finalize_dataset(strict_no_overlap(), "strict_no_overlap"),
    "low_overlap_only": finalize_dataset(low_overlap_only(), "low_overlap_only"),
}

for name, data in DATASETS.items():
    print("\n==", name, data.shape)
    print(data.groupby(["binary_label_name", "final_class", "window_policy"]).size().to_string())
    print("overlap rows:", int(((data["final_class"] != "normal") & (overlap_count(data) > 0)).sum()))



== reference_pre_event_all (181, 194)
binary_label_name  final_class  window_policy  
event              activity     activity_pre_7d    47
                   fault        fault_pre_7d       62
                   task         task_pre_7d        37
normal             normal       normal_event_7d    35
overlap rows: 38

== reference_low_overlap_hybrid (185, 194)
binary_label_name  final_class  window_policy  
event              activity     activity_pre_7d    47
                   fault        fault_pre_7d       62
                   task         task_post_7d       41
normal             normal       normal_event_7d    35
overlap rows: 15

== strict_no_overlap (173, 194)
binary_label_name  final_class  window_policy   
event              activity     activity_post_7d     3
                                activity_pre_7d     43
                   fault        fault_pre_7d        55
                   task         task_post_7d        37
normal             normal       normal_event_7d     3

In [5]:

policy_rows = []
for source_name, data in DATASETS.items():
    for class_name, group in data.groupby("final_class"):
        policy_rows.append({
            "dataset": source_name,
            "final_class": class_name,
            "rows": len(group),
            "substation_count": int(group["substation_id"].nunique()),
            "window_policies": "|".join(sorted(group["window_policy"].dropna().unique())),
            "coverage_min": float(group["coverage_rate"].min()),
            "coverage_median": float(group["coverage_rate"].median()),
            "overlap_rows": int((overlap_count(group) > 0).sum()),
        })
policy_audit = pd.DataFrame(policy_rows)
policy_audit.to_csv(OUT_POLICY_AUDIT, index=False, encoding="utf-8-sig")

summary_rows = []
for dataset_name, data in DATASETS.items():
    summary_rows.append({
        "dataset": dataset_name,
        "evaluation_rows": len(data),
        "normal_rows": int((data["binary_label"] == 0).sum()),
        "event_rows": int((data["binary_label"] == 1).sum()),
        "fault_rows": int((data["final_class"] == "fault").sum()),
        "task_rows": int((data["final_class"] == "task").sum()),
        "activity_rows": int((data["final_class"] == "activity").sum()),
        "substation_count": int(data["substation_id"].nunique()),
        "event_overlap_rows": int(((data["final_class"] != "normal") & (overlap_count(data) > 0)).sum()),
        "coverage_min": float(data["coverage_rate"].min()),
        "coverage_median": float(data["coverage_rate"].median()),
        "train_candidate_normal_rows": 0,
        "note": "evaluation dataset",
    })

candidate_normal_rows = int(candidate_pool["label_strength"].eq("candidate_normal").sum()) if not candidate_pool.empty else 0
summary_rows.append({
    "dataset": "train_normal_expanded",
    "evaluation_rows": len(DATASETS["reference_pre_event_all"]),
    "normal_rows": int((DATASETS["reference_pre_event_all"]["binary_label"] == 0).sum()),
    "event_rows": int((DATASETS["reference_pre_event_all"]["binary_label"] == 1).sum()),
    "fault_rows": int((DATASETS["reference_pre_event_all"]["final_class"] == "fault").sum()),
    "task_rows": int((DATASETS["reference_pre_event_all"]["final_class"] == "task").sum()),
    "activity_rows": int((DATASETS["reference_pre_event_all"]["final_class"] == "activity").sum()),
    "substation_count": int(DATASETS["reference_pre_event_all"]["substation_id"].nunique()),
    "event_overlap_rows": int(((DATASETS["reference_pre_event_all"]["final_class"] != "normal") & (overlap_count(DATASETS["reference_pre_event_all"]) > 0)).sum()),
    "coverage_min": float(DATASETS["reference_pre_event_all"]["coverage_rate"].min()),
    "coverage_median": float(DATASETS["reference_pre_event_all"]["coverage_rate"].median()),
    "train_candidate_normal_rows": candidate_normal_rows,
    "note": "reference_pre_event_all evaluation; candidate normal is train-fold only and compact13 only",
})

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(OUT_DATASET_SUMMARY, index=False, encoding="utf-8-sig")
dataset_summary


,dataset,evaluation_rows,normal_rows,event_rows,fault_rows,task_rows,activity_rows,substation_count,event_overlap_rows,coverage_min,coverage_median,train_candidate_normal_rows,note
0,reference_pre_event_all,181,35,146,62,37,47,33,38,0.995040,1.0,0,evaluation dataset
1,reference_low_overlap_hybrid,185,35,150,62,41,47,33,15,0.982143,1.0,0,evaluation dataset
2,strict_no_overlap,173,35,138,55,37,46,33,0,0.982143,1.0,0,evaluation dataset
3,low_overlap_only,170,35,135,55,37,43,33,0,0.982143,1.0,0,evaluation dataset
4,train_normal_expanded,181,35,146,62,37,47,33,38,0.995040,1.0,70,reference_pre_event_all evaluation; candidate ...


## Model Evaluation

각 dataset/feature/model을 동일한 group CV 방식으로 평가한다.

In [6]:

MODEL_NAMES = [
    "dummy_most_frequent",
    "logistic_balanced",
    "random_forest_balanced_depth3",
    "extra_trees_balanced_depth3",
]


def build_model(model_name: str):
    if model_name == "dummy_most_frequent":
        return DummyClassifier(strategy="most_frequent")
    if model_name == "logistic_balanced":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)),
        ])
    if model_name == "random_forest_balanced_depth3":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == "extra_trees_balanced_depth3":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", ExtraTreesClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    raise ValueError(model_name)


def clean_feature_matrix(data: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
    return X.replace([np.inf, -np.inf], np.nan)


def get_event_probability(model, X: pd.DataFrame) -> np.ndarray:
    proba = model.predict_proba(X)
    classes = list(model.classes_) if hasattr(model, "classes_") else list(model[-1].classes_)
    return proba[:, classes.index(1)]


def metric_row(y_true, y_prob, threshold: float) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan
    return {
        "threshold": threshold,
        "rows": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": auc,
        "precision_event": precision_score(y_true, y_pred, zero_division=0),
        "recall_event": recall_score(y_true, y_pred, zero_division=0),
        "f1_event": f1_score(y_true, y_pred, zero_division=0),
        "normal_fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def candidate_normal_frame(feature_cols: list[str]) -> pd.DataFrame:
    if candidate_pool.empty:
        return pd.DataFrame()
    required = set(feature_cols) | {"label_strength", "substation_id"}
    if not required.issubset(candidate_pool.columns):
        return pd.DataFrame()
    cand = candidate_pool.loc[candidate_pool["label_strength"].eq("candidate_normal")].copy()
    cand["binary_label"] = 0
    cand["binary_label_name"] = "normal"
    cand["event_type_binary"] = "candidate_normal"
    cand["final_class"] = "candidate_normal"
    cand["source_id"] = cand["sample_id"]
    cand["disturbance_row_id"] = np.nan
    cand["hard_normal_tag"] = cand.get("review_tag", np.nan)
    cand["overlap_disturbance_count"] = 0
    cand["coverage_eligible"] = cand["coverage_rate"] >= 0.95
    return cand.loc[cand["coverage_eligible"]].copy()

print("models:", MODEL_NAMES)


models: ['dummy_most_frequent', 'logistic_balanced', 'random_forest_balanced_depth3', 'extra_trees_balanced_depth3']


In [7]:

def evaluate_dataset(dataset_name: str, data: pd.DataFrame, feature_set_name: str, feature_cols: list[str], train_extra: pd.DataFrame | None = None):
    rows_pred = []
    rows_threshold = []
    rows_fold = []
    y = data["binary_label"].astype(int).to_numpy()
    groups = data["substation_id"].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    X = clean_feature_matrix(data, feature_cols)
    splits = list(splitter.split(data, y, groups))
    extra = train_extra if train_extra is not None else pd.DataFrame()

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])
        overlap = sorted(train_groups & test_groups)
        assert not overlap, f"group leakage: {dataset_name} {feature_set_name} fold {fold}"

        X_train = X.iloc[train_idx].copy()
        y_train = y[train_idx].copy()
        extra_rows_used = 0
        if not extra.empty:
            allowed_extra = extra.loc[~extra["substation_id"].astype(int).isin(test_groups)].copy()
            if not allowed_extra.empty:
                X_extra = clean_feature_matrix(allowed_extra, feature_cols)
                X_train = pd.concat([X_train, X_extra], ignore_index=True)
                y_train = np.concatenate([y_train, allowed_extra["binary_label"].astype(int).to_numpy()])
                extra_rows_used = len(allowed_extra)

        X_test = X.iloc[test_idx]
        y_test = y[test_idx]
        test_data = data.iloc[test_idx].copy()

        for model_name in MODEL_NAMES:
            model = build_model(model_name)
            model.fit(X_train, y_train)
            y_prob = get_event_probability(model, X_test)

            pred_cols = ["source_id", "final_class", "event_type_binary", "source_event_id", "disturbance_row_id", "substation_id", "window_policy", "window_start", "window_end", "coverage_rate", "overlap_disturbance_count", "hard_normal_tag"]
            pred_frame = test_data[[col for col in pred_cols if col in test_data.columns]].copy()
            pred_frame["dataset"] = dataset_name
            pred_frame["feature_set"] = feature_set_name
            pred_frame["feature_count"] = len(feature_cols)
            pred_frame["model"] = model_name
            pred_frame["fold"] = fold
            pred_frame["y_true"] = y_test
            pred_frame["probability_event"] = y_prob
            pred_frame["train_extra_rows_used"] = extra_rows_used
            for threshold in THRESHOLDS:
                pred_frame[f"prediction_t{str(threshold).replace('.', '_')}"] = (y_prob >= threshold).astype(int)
            rows_pred.extend(pred_frame.to_dict("records"))

            for threshold in THRESHOLDS:
                row = metric_row(y_test, y_prob, threshold)
                row.update({
                    "dataset": dataset_name,
                    "feature_set": feature_set_name,
                    "feature_count": len(feature_cols),
                    "model": model_name,
                    "fold": fold,
                    "metric_scope": "fold",
                    "train_rows": int(len(train_idx)),
                    "train_extra_rows_used": int(extra_rows_used),
                    "test_rows": int(len(test_idx)),
                    "train_groups": "|".join(map(str, sorted(train_groups))),
                    "test_groups": "|".join(map(str, sorted(test_groups))),
                    "group_overlap_count": 0,
                })
                rows_threshold.append(row)
                if threshold == DEFAULT_THRESHOLD:
                    rows_fold.append(row.copy())
    return rows_pred, rows_threshold, rows_fold


prediction_rows = []
threshold_rows = []
cv_metric_rows = []

for dataset_name, data in DATASETS.items():
    for feature_set_name, feature_cols in FEATURE_COLUMNS.items():
        pred, thr, cv = evaluate_dataset(dataset_name, data, feature_set_name, feature_cols)
        prediction_rows.extend(pred)
        threshold_rows.extend(thr)
        cv_metric_rows.extend(cv)

extra_compact13 = candidate_normal_frame(FEATURE_COLUMNS["compact13"])
if not extra_compact13.empty:
    pred, thr, cv = evaluate_dataset(
        "train_normal_expanded",
        DATASETS["reference_pre_event_all"],
        "compact13",
        FEATURE_COLUMNS["compact13"],
        train_extra=extra_compact13,
    )
    prediction_rows.extend(pred)
    threshold_rows.extend(thr)
    cv_metric_rows.extend(cv)

predictions = pd.DataFrame(prediction_rows)
threshold_metrics_fold = pd.DataFrame(threshold_rows)
cv_metrics_fold = pd.DataFrame(cv_metric_rows)
print("predictions", predictions.shape)
print("threshold fold metrics", threshold_metrics_fold.shape)
print("cv fold metrics", cv_metrics_fold.shape)


predictions (6396, 23)
threshold fold metrics (540, 25)
cv fold metrics (180, 25)


In [8]:

aggregate_rows = []
for keys, group in predictions.groupby(["dataset", "feature_set", "feature_count", "model"], dropna=False):
    dataset_name, feature_set_name, feature_count, model_name = keys
    y_true = group["y_true"].astype(int).to_numpy()
    y_prob = group["probability_event"].astype(float).to_numpy()
    for threshold in THRESHOLDS:
        row = metric_row(y_true, y_prob, threshold)
        row.update({
            "dataset": dataset_name,
            "feature_set": feature_set_name,
            "feature_count": int(feature_count),
            "model": model_name,
            "fold": "all",
            "metric_scope": "aggregate",
            "train_rows": np.nan,
            "train_extra_rows_used": int(group["train_extra_rows_used"].max()) if "train_extra_rows_used" in group.columns else 0,
            "test_rows": int(len(group)),
            "train_groups": np.nan,
            "test_groups": np.nan,
            "group_overlap_count": 0,
        })
        aggregate_rows.append(row)

threshold_metrics_agg = pd.DataFrame(aggregate_rows)
threshold_metrics = pd.concat([threshold_metrics_fold, threshold_metrics_agg], ignore_index=True)
cv_metrics = pd.concat([
    cv_metrics_fold,
    threshold_metrics_agg.loc[threshold_metrics_agg["threshold"].eq(DEFAULT_THRESHOLD)].copy(),
], ignore_index=True)

predictions.to_csv(OUT_PREDICTIONS, index=False, encoding="utf-8-sig")
threshold_metrics.to_csv(OUT_THRESHOLD_METRICS, index=False, encoding="utf-8-sig")
cv_metrics.to_csv(OUT_CV_METRICS, index=False, encoding="utf-8-sig")
threshold_metrics_agg.sort_values(["threshold", "balanced_accuracy"], ascending=[True, False]).head(20)


,threshold,rows,accuracy,balanced_accuracy,roc_auc,precision_event,recall_event,f1_event,normal_fpr,tn,...,feature_count,model,fold,metric_scope,train_rows,train_extra_rows_used,test_rows,train_groups,test_groups,group_overlap_count
18,0.4,170,0.776471,0.689947,0.784974,0.875969,0.837037,0.856061,0.457143,19,...,154,logistic_balanced,all,aggregate,NaN,0,170,NaN,NaN,0
78,0.4,173,0.722543,0.655487,0.722153,0.868852,0.768116,0.815385,0.457143,19,...,13,logistic_balanced,all,aggregate,NaN,0,173,NaN,NaN,0
54,0.4,181,0.712707,0.648141,0.700196,0.873016,0.753425,0.808824,0.457143,19,...,13,logistic_balanced,all,aggregate,NaN,0,181,NaN,NaN,0
6,0.4,170,0.705882,0.645503,0.695238,0.863248,0.748148,0.801587,0.457143,19,...,13,logistic_balanced,all,aggregate,NaN,0,170,NaN,NaN,0
90,0.4,173,0.739884,0.645031,0.731263,0.860465,0.804348,0.831461,0.514286,17,...,154,logistic_balanced,all,aggregate,NaN,0,173,NaN,NaN,0
9,0.4,170,0.805882,0.644974,0.704339,0.849315,0.918519,0.882562,0.628571,13,...,13,random_forest_balanced_depth3,all,aggregate,NaN,0,170,NaN,NaN,0
105,0.4,181,0.734807,0.640117,0.719961,0.865672,0.794521,0.828571,0.514286,17,...,13,random_forest_balanced_depth3,all,aggregate,NaN,60,181,NaN,NaN,0
42,0.4,185,0.767568,0.637619,0.787238,0.863946,0.846667,0.855219,0.571429,15,...,154,logistic_balanced,all,aggregate,NaN,0,185,NaN,NaN,0
30,0.4,185,0.729730,0.636190,0.723810,0.867647,0.786667,0.825175,0.514286,17,...,13,logistic_balanced,all,aggregate,NaN,0,185,NaN,NaN,0
102,0.4,181,0.690608,0.623581,0.716243,0.862903,0.732877,0.792593,0.485714,18,...,13,logistic_balanced,all,aggregate,NaN,60,181,NaN,NaN,0


## Decision Matrix And Error Audit

In [9]:

agg = threshold_metrics_agg.copy()
dummy_lookup = (
    agg.loc[agg["model"].eq("dummy_most_frequent"), ["dataset", "feature_set", "threshold", "balanced_accuracy"]]
    .rename(columns={"balanced_accuracy": "dummy_balanced_accuracy"})
)
decision_matrix = agg.merge(dummy_lookup, on=["dataset", "feature_set", "threshold"], how="left")
decision_matrix["balanced_accuracy_lift_vs_dummy"] = decision_matrix["balanced_accuracy"] - decision_matrix["dummy_balanced_accuracy"]
decision_matrix["passes_event_recall"] = decision_matrix["recall_event"] >= 0.80
decision_matrix["passes_normal_fpr"] = decision_matrix["normal_fpr"] <= 0.25
decision_matrix["passes_dummy_lift"] = decision_matrix["balanced_accuracy_lift_vs_dummy"] >= 0.15
decision_matrix["passes_group_overlap"] = decision_matrix["group_overlap_count"].fillna(0).eq(0)
decision_matrix["passes_gate_criteria"] = (
    decision_matrix["passes_event_recall"]
    & decision_matrix["passes_normal_fpr"]
    & decision_matrix["passes_dummy_lift"]
    & decision_matrix["passes_group_overlap"]
    & ~decision_matrix["model"].eq("dummy_most_frequent")
)


def decision_candidate(row: pd.Series) -> str:
    if row["passes_gate_criteria"]:
        return "normal_vs_event_gate_candidate_locked"
    if row["passes_event_recall"] or row["passes_normal_fpr"]:
        return "gate_policy_iteration_needed"
    return "normal_vs_event_gate_unstable_keep_pre_event_binary"


decision_matrix["decision_candidate"] = decision_matrix.apply(decision_candidate, axis=1)


def overall_decision_at_default(frame: pd.DataFrame) -> str:
    default = frame.loc[frame["threshold"].eq(DEFAULT_THRESHOLD) & ~frame["model"].eq("dummy_most_frequent")].copy()
    if default["passes_gate_criteria"].any():
        return "normal_vs_event_gate_candidate_locked"
    reference_best = default.loc[default["dataset"].eq("reference_pre_event_all"), "balanced_accuracy"].max()
    non_reference_best = default.loc[~default["dataset"].eq("reference_pre_event_all"), "balanced_accuracy"].max()
    if pd.notna(non_reference_best) and pd.notna(reference_best) and non_reference_best >= reference_best - 0.02:
        return "gate_policy_iteration_needed"
    return "normal_vs_event_gate_unstable_keep_pre_event_binary"


overall_decision = overall_decision_at_default(decision_matrix)
decision_matrix["overall_decision_at_t0_5"] = overall_decision
decision_matrix.to_csv(OUT_DECISION_MATRIX, index=False, encoding="utf-8-sig")
print("overall_decision_at_t0_5:", overall_decision)
decision_matrix.sort_values(["threshold", "passes_gate_criteria", "balanced_accuracy"], ascending=[True, False, False]).head(20)


overall_decision_at_t0_5: gate_policy_iteration_needed


,threshold,rows,accuracy,balanced_accuracy,roc_auc,precision_event,recall_event,f1_event,normal_fpr,tn,...,group_overlap_count,dummy_balanced_accuracy,balanced_accuracy_lift_vs_dummy,passes_event_recall,passes_normal_fpr,passes_dummy_lift,passes_group_overlap,passes_gate_criteria,decision_candidate,overall_decision_at_t0_5
18,0.4,170,0.776471,0.689947,0.784974,0.875969,0.837037,0.856061,0.457143,19,...,0,0.5,0.189947,True,False,True,True,False,gate_policy_iteration_needed,gate_policy_iteration_needed
78,0.4,173,0.722543,0.655487,0.722153,0.868852,0.768116,0.815385,0.457143,19,...,0,0.5,0.155487,False,False,True,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed
54,0.4,181,0.712707,0.648141,0.700196,0.873016,0.753425,0.808824,0.457143,19,...,0,0.5,0.148141,False,False,False,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed
6,0.4,170,0.705882,0.645503,0.695238,0.863248,0.748148,0.801587,0.457143,19,...,0,0.5,0.145503,False,False,False,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed
90,0.4,173,0.739884,0.645031,0.731263,0.860465,0.804348,0.831461,0.514286,17,...,0,0.5,0.145031,True,False,False,True,False,gate_policy_iteration_needed,gate_policy_iteration_needed
9,0.4,170,0.805882,0.644974,0.704339,0.849315,0.918519,0.882562,0.628571,13,...,0,0.5,0.144974,True,False,False,True,False,gate_policy_iteration_needed,gate_policy_iteration_needed
105,0.4,181,0.734807,0.640117,0.719961,0.865672,0.794521,0.828571,0.514286,17,...,0,0.5,0.140117,False,False,False,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed
42,0.4,185,0.767568,0.637619,0.787238,0.863946,0.846667,0.855219,0.571429,15,...,0,0.5,0.137619,True,False,False,True,False,gate_policy_iteration_needed,gate_policy_iteration_needed
30,0.4,185,0.729730,0.636190,0.723810,0.867647,0.786667,0.825175,0.514286,17,...,0,0.5,0.136190,False,False,False,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed
102,0.4,181,0.690608,0.623581,0.716243,0.862903,0.732877,0.792593,0.485714,18,...,0,0.5,0.123581,False,False,False,True,False,normal_vs_event_gate_unstable_keep_pre_event_b...,gate_policy_iteration_needed


In [10]:

error_rows = []
for _, row in decision_matrix.loc[~decision_matrix["model"].eq("dummy_most_frequent")].iterrows():
    threshold = row["threshold"]
    pred_col = f"prediction_t{str(threshold).replace('.', '_')}"
    subset = predictions.loc[
        predictions["dataset"].eq(row["dataset"])
        & predictions["feature_set"].eq(row["feature_set"])
        & predictions["model"].eq(row["model"])
    ].copy()
    if pred_col not in subset.columns:
        continue
    subset["prediction"] = subset[pred_col].astype(int)
    subset["error_type"] = np.select(
        [subset["y_true"].eq(0) & subset["prediction"].eq(1), subset["y_true"].eq(1) & subset["prediction"].eq(0)],
        ["FP", "FN"],
        default="correct",
    )
    errors = subset.loc[subset["error_type"].isin(["FP", "FN"])].copy()
    errors["threshold"] = threshold
    errors["balanced_accuracy"] = row["balanced_accuracy"]
    errors["recall_event"] = row["recall_event"]
    errors["normal_fpr"] = row["normal_fpr"]
    error_rows.extend(errors.to_dict("records"))

error_audit = pd.DataFrame(error_rows)
error_audit.to_csv(OUT_ERROR_AUDIT, index=False, encoding="utf-8-sig")
print("error audit", error_audit.shape)
error_audit.groupby(["dataset", "feature_set", "model", "threshold", "error_type"]).size().reset_index(name="rows").head(20)


error audit (4843, 29)


,dataset,feature_set,model,threshold,error_type,rows
0,low_overlap_only,compact13,extra_trees_balanced_depth3,0.4,FN,1
1,low_overlap_only,compact13,extra_trees_balanced_depth3,0.4,FP,35
2,low_overlap_only,compact13,extra_trees_balanced_depth3,0.5,FN,41
3,low_overlap_only,compact13,extra_trees_balanced_depth3,0.5,FP,11
4,low_overlap_only,compact13,extra_trees_balanced_depth3,0.6,FN,108
5,low_overlap_only,compact13,extra_trees_balanced_depth3,0.6,FP,2
6,low_overlap_only,compact13,logistic_balanced,0.4,FN,34
7,low_overlap_only,compact13,logistic_balanced,0.4,FP,16
8,low_overlap_only,compact13,logistic_balanced,0.5,FN,48
9,low_overlap_only,compact13,logistic_balanced,0.5,FP,13


## Validation Checks

In [11]:

required_outputs = [
    OUT_POLICY_AUDIT,
    OUT_DATASET_SUMMARY,
    OUT_CV_METRICS,
    OUT_PREDICTIONS,
    OUT_THRESHOLD_METRICS,
    OUT_ERROR_AUDIT,
    OUT_DECISION_MATRIX,
]
for path in required_outputs:
    assert path.exists(), path

assert int(dataset_summary.loc[dataset_summary["dataset"].eq("reference_pre_event_all"), "normal_rows"].iloc[0]) == 35
assert int(dataset_summary.loc[dataset_summary["dataset"].eq("reference_low_overlap_hybrid"), "normal_rows"].iloc[0]) == 35
assert int(dataset_summary.loc[dataset_summary["dataset"].eq("strict_no_overlap"), "event_overlap_rows"].iloc[0]) == 0
assert int(dataset_summary.loc[dataset_summary["dataset"].eq("low_overlap_only"), "event_overlap_rows"].iloc[0]) == 0
assert len(FEATURE_COLUMNS["compact13"]) == 13
assert len(FEATURE_COLUMNS["expanded154"]) == 154
assert threshold_metrics["group_overlap_count"].fillna(0).max() == 0

for path in required_outputs:
    text = path.read_text(encoding="utf-8-sig", errors="ignore").lower()
    assert ("manufacturer" + "_2") not in text
    assert ("manufacturer" + " " + "2") not in text

print("validation checks passed")


validation checks passed


## Results

In [12]:
default_view = (
    decision_matrix.loc[decision_matrix["threshold"].eq(DEFAULT_THRESHOLD) & ~decision_matrix["model"].eq("dummy_most_frequent")]
    .sort_values(["passes_gate_criteria", "balanced_accuracy", "recall_event"], ascending=[False, False, False])
)
default_view[["dataset", "feature_set", "model", "balanced_accuracy", "balanced_accuracy_lift_vs_dummy", "recall_event", "normal_fpr", "fp", "fn", "passes_gate_criteria", "decision_candidate"]].head(15)

,dataset,feature_set,model,balanced_accuracy,balanced_accuracy_lift_vs_dummy,recall_event,normal_fpr,fp,fn,passes_gate_criteria,decision_candidate
19,low_overlap_only,expanded154,logistic_balanced,0.732275,0.232275,0.807407,0.342857,12,26,False,gate_policy_iteration_needed
70,reference_pre_event_all,expanded154,random_forest_balanced_depth3,0.725832,0.225832,0.794521,0.342857,12,30,False,normal_vs_event_gate_unstable_keep_pre_event_b...
52,reference_pre_event_all,compact13,extra_trees_balanced_depth3,0.707632,0.207632,0.643836,0.228571,8,52,False,gate_policy_iteration_needed
40,reference_low_overlap_hybrid,expanded154,extra_trees_balanced_depth3,0.703810,0.203810,0.493333,0.085714,3,76,False,gate_policy_iteration_needed
43,reference_low_overlap_hybrid,expanded154,logistic_balanced,0.699048,0.199048,0.826667,0.428571,15,26,False,gate_policy_iteration_needed
100,train_normal_expanded,compact13,extra_trees_balanced_depth3,0.697358,0.197358,0.623288,0.228571,8,55,False,gate_policy_iteration_needed
64,reference_pre_event_all,expanded154,extra_trees_balanced_depth3,0.696869,0.196869,0.479452,0.085714,3,76,False,gate_policy_iteration_needed
4,low_overlap_only,compact13,extra_trees_balanced_depth3,0.691005,0.191005,0.696296,0.314286,11,41,False,normal_vs_event_gate_unstable_keep_pre_event_b...
46,reference_low_overlap_hybrid,expanded154,random_forest_balanced_depth3,0.685714,0.185714,0.800000,0.428571,15,30,False,gate_policy_iteration_needed
16,low_overlap_only,expanded154,extra_trees_balanced_depth3,0.683598,0.183598,0.481481,0.114286,4,70,False,gate_policy_iteration_needed


In [13]:
threshold_view = (
    decision_matrix.loc[~decision_matrix["model"].eq("dummy_most_frequent")]
    .sort_values(["threshold", "passes_gate_criteria", "balanced_accuracy"], ascending=[True, False, False])
    .groupby("threshold")
    .head(8)
)
threshold_view[["threshold", "dataset", "feature_set", "model", "balanced_accuracy", "recall_event", "normal_fpr", "fp", "fn", "passes_gate_criteria"]].head(24)

,threshold,dataset,feature_set,model,balanced_accuracy,recall_event,normal_fpr,fp,fn,passes_gate_criteria
18,0.4,low_overlap_only,expanded154,logistic_balanced,0.689947,0.837037,0.457143,16,22,False
78,0.4,strict_no_overlap,compact13,logistic_balanced,0.655487,0.768116,0.457143,16,32,False
54,0.4,reference_pre_event_all,compact13,logistic_balanced,0.648141,0.753425,0.457143,16,36,False
6,0.4,low_overlap_only,compact13,logistic_balanced,0.645503,0.748148,0.457143,16,34,False
90,0.4,strict_no_overlap,expanded154,logistic_balanced,0.645031,0.804348,0.514286,18,27,False
9,0.4,low_overlap_only,compact13,random_forest_balanced_depth3,0.644974,0.918519,0.628571,22,11,False
105,0.4,train_normal_expanded,compact13,random_forest_balanced_depth3,0.640117,0.794521,0.514286,18,30,False
42,0.4,reference_low_overlap_hybrid,expanded154,logistic_balanced,0.637619,0.846667,0.571429,20,23,False
19,0.5,low_overlap_only,expanded154,logistic_balanced,0.732275,0.807407,0.342857,12,26,False
70,0.5,reference_pre_event_all,expanded154,random_forest_balanced_depth3,0.725832,0.794521,0.342857,12,30,False


## Write Report

In [14]:

default_view = (
    decision_matrix.loc[decision_matrix["threshold"].eq(DEFAULT_THRESHOLD) & ~decision_matrix["model"].eq("dummy_most_frequent")]
    .sort_values(["passes_gate_criteria", "balanced_accuracy", "recall_event"], ascending=[False, False, False])
)
threshold_view = (
    decision_matrix.loc[~decision_matrix["model"].eq("dummy_most_frequent")]
    .sort_values(["threshold", "passes_gate_criteria", "balanced_accuracy"], ascending=[True, False, False])
    .groupby("threshold")
    .head(8)
)


def fmt_float(value, digits=4):
    if pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"


def md_table(df: pd.DataFrame) -> str:
    table = df.copy()
    for col in table.columns:
        table[col] = table[col].map(lambda x: "" if pd.isna(x) else str(x))
    header = "| " + " | ".join(table.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in table.astype(str).values.tolist()]
    return "\n".join([header, sep] + body)


best_default = default_view.iloc[0]
if overall_decision == "normal_vs_event_gate_candidate_locked":
    conclusion = "threshold 0.5 기준에서 채택 조건을 만족한 gate 후보가 있다."
elif overall_decision == "gate_policy_iteration_needed":
    conclusion = "threshold 0.5 기준에서 최종 잠금은 어렵지만, overlap 제거/normal 증강 후보가 기준선과 비슷한 성능을 보여 추가 기준 반복이 필요하다."
else:
    conclusion = "threshold 0.5 기준에서 기준선보다 안정적인 후보를 찾지 못했다. normal vs event gate보다 기존 pre_event binary를 유지하는 것이 현실적이다."

if default_view["passes_event_recall"].any() and default_view["passes_normal_fpr"].any():
    bottleneck = "recall 우수 조합과 FPR 우수 조합이 서로 다르다."
elif default_view["passes_event_recall"].any():
    bottleneck = "event recall은 일부 조합에서 맞지만 normal FPR이 과도하다."
elif default_view["passes_normal_fpr"].any():
    bottleneck = "normal FPR은 일부 조합에서 맞지만 event recall이 부족하다."
else:
    bottleneck = "recall과 FPR 모두 기준을 동시에 만족하지 못했다."

show_cols = ["dataset", "feature_set", "model", "balanced_accuracy", "balanced_accuracy_lift_vs_dummy", "recall_event", "normal_fpr", "fp", "fn", "passes_gate_criteria", "decision_candidate"]
threshold_cols = ["threshold", "dataset", "feature_set", "model", "balanced_accuracy", "recall_event", "normal_fpr", "fp", "fn", "passes_gate_criteria"]

lines = [
    "# M1 Normal vs Event Gate Policy Iteration 보고서",
    "",
    "## 결론",
    f"- 최종 판단: `{overall_decision}`",
    f"- 해석: {conclusion}",
    f"- 최고 기본 threshold 조합: `{best_default['dataset']}` / `{best_default['feature_set']}` / `{best_default['model']}`",
    f"- 최고 기본 threshold 성능: balanced accuracy `{fmt_float(best_default['balanced_accuracy'])}`, event recall `{fmt_float(best_default['recall_event'])}`, normal FPR `{fmt_float(best_default['normal_fpr'])}`",
    f"- 주요 병목: {bottleneck}",
    "",
    "## Dataset 구성",
    md_table(dataset_summary),
    "",
    "## Policy Audit",
    md_table(policy_audit),
    "",
    "## Threshold 0.5 주요 결과",
    md_table(default_view[show_cols].head(20)),
    "",
    "## Threshold 참고 결과",
    md_table(threshold_view[threshold_cols].head(24)),
    "",
    "## 진단",
    "- label 기준: `event=fault+task+activity`로 유지했지만, event 내부 패턴 차이가 커서 단일 gate가 흔들린다.",
    "- window overlap: strict/low-overlap 후보를 따로 검증했으나 threshold 0.5 채택 조건을 만족하지 못했다.",
    "- normal 샘플: candidate normal train 증강은 compact13에서만 가능하며, 평가 normal 35건은 변경하지 않았다.",
    "- feature/model: expanded154와 tree 모델이 기준선에서는 가장 강하지만, 오탐과 미탐 tradeoff가 아직 크다.",
    "",
    "## 검증 사항",
    "- 모든 계산은 M1 taxonomy 산출물 기준으로 수행했다.",
    "- normal 35건 label은 변경하지 않았다.",
    "- 학습 입력에서 metadata/date/event id/substation/coverage는 제외했다.",
    "- group CV train/test substation overlap은 모든 fold에서 0이다.",
    "- 비대상 제조사 문자열/경로/결과는 새 산출물에 포함하지 않았다.",
    "",
    "## 다음 단계",
]
if overall_decision == "normal_vs_event_gate_candidate_locked":
    lines.append("- 잠긴 gate 후보를 기준으로 event 내부 `fault/task/activity` 3분류 검증으로 진행한다.")
elif overall_decision == "gate_policy_iteration_needed":
    lines.append("- 바로 3분류로 가지 말고, event 내부를 한 번에 묶는 gate 대신 fault/task/activity별 one-vs-normal gate를 비교한다.")
else:
    lines.append("- 4분류 gate를 강행하지 말고, 기존 `normal vs pre_event` binary를 유지하면서 fault/task/activity별 개별 binary 가능성을 먼저 본다.")

OUT_REPORT.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(OUT_REPORT)


C:\3rd_Project\HeatGridAgent\07_데이터산출물\18_M1_normal_vs_event_gate_policy_iteration_보고서.md


## Takeaways

보고서와 decision matrix를 기준으로 `normal vs event` gate를 잠글지, 아니면 class별 binary gate로 분해할지 판단한다.